# NER 第 1 步：加载与验收 CoNLL-2003

版本：2026-11-01 data-v1。**代码已准备；真实数据验收需要你运行后确认。**

沿用第 0 步的 Colab Enterprise 运行时和本地目录。此笔记本自行初始化变量，
不要求另一本 notebook 的 Python 全局变量存在。若使用新运行时，先完成第 0 步环境准备。
本步不需要 Drive、bucket、Google API 密钥或 OAuth JSON。

运行顺序：**1.1 → 1.2 → 1.3 → 1.4 → 1.5 → 下载 ZIP → 1.6 确认**。
保留原来的第 0 步 ZIP；本步 ZIP 保存代码、来源、标签映射和验收记录，数据保存在运行时。
同时用「文件 → 下载 .ipynb」保存当前笔记本。

验收目标：train / validation / test = **14041 / 3250 / 3453**；从元数据解码标签；
逐句检查长度、标签值、IOB2、DOCSTART 和句序；展示训练集样例。
没有训练、调参或测试集评分。完整 EDA 属于下一步。

来源说明：开发文档首选 `eriktks/conll2003`，并明确允许 Parquet 副本作为兼容方案。
`datasets==4.0.0` 已移除旧式加载脚本，所以使用 `tomaarsen/conll2003` 的固定提交
`3a0c0677d3177b6122fceed3239dda7a5de6a514`，直接读三个 Parquet 文件。
保留该副本的 document_id / sentence_id 辅助列；不打乱、过滤或转换标签。
结构校验不等于已证明与 eriktks 仓库逐字节相同，记录会明确这一限制。

参考：[datasets 4.0.0 说明](https://github.com/huggingface/datasets/releases/tag/4.0.0) ·
[实际数据来源与元数据](https://huggingface.co/datasets/tomaarsen/conll2003/blob/main/README.md) ·
[开发文档指定数据卡](https://huggingface.co/datasets/eriktks/conll2003)


## 1.1 检查依赖
第 0 步已安装 `datasets==4.0.0`，通常直接通过。如果缺失或版本不同，本单元只安装该固定版本；
看到重启提示后先重启 Python 会话，再从 1.1 开始运行。无需重新配置云端授权。


In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

try:
    installed_datasets = metadata.version('datasets')
except metadata.PackageNotFoundError:
    installed_datasets = None
if installed_datasets != '4.0.0':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'datasets==4.0.0'])
    raise RuntimeError('Installation finished. Restart the Python session, then rerun 1.1.')
import datasets
if datasets.__version__ != '4.0.0':
    raise RuntimeError('A different datasets version is still imported. Restart the Python session.')
print('Python:', sys.version.split()[0])
print('datasets:', datasets.__version__)
print('Dependencies ready. Continue to 1.2.')


## 1.2 初始化本步目录和工具
默认复用第 0 步目录。如果你把备份恢复到其他目录，只修改下面的 `PROJECT_ROOT`。
每次重新初始化都会产生新 run_id；已存在的第 0 步日志会保留。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import json
import platform
import random
import uuid

NOTEBOOK_REVISION = '2026-11-01-data-v1'
SEED = 42
random.seed(SEED)
PROJECT_ROOT = Path.home() / 'cs5489_ner_local'
EXPORT_DIR = Path.home() / 'ner_exports'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
STEP1_COMPLETE = DATA_LOADED = DATA_VERIFIED = DATA_SAVED = BACKUP_VERIFIED = BACKUP_DOWNLOADED = False
for folder in ['config', 'logs', 'src', 'results', 'figures', 'predictions', 'data']:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT_ROOT)
print('Exports:', EXPORT_DIR)
print('Run:', RUN_ID)


In [ ]:
SOURCE_BUNDLE = {'local_backup.py': '"""Verified, portable NER backups. Uses local files and the Python standard library."""\nimport hashlib\nimport json\nimport re\nimport shutil\nimport stat\nimport uuid\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\n\nFORMAT = \'cs5489-ner-local-backup-v1\'\nALLOWED_FOLDERS = {\'config\', \'logs\', \'src\', \'results\', \'figures\', \'predictions\', \'notebooks\'}\nALLOWED_ROOT_FILES = {\'README.md\', \'requirements.txt\', \'requirements-step0.txt\'}\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _relative_path(value):\n    if not isinstance(value, str) or not value or \'\\\\\' in value or \':\' in value:\n        raise ValueError(\'Invalid backup member path.\')\n    parts = value.split(\'/\')\n    if any(part in (\'\', \'.\', \'..\') or part.startswith(\'.\') for part in parts):\n        raise ValueError(\'Unsafe backup member path.\')\n    if PurePosixPath(value).is_absolute():\n        raise ValueError(\'Absolute backup member paths are forbidden.\')\n    return PurePosixPath(value)\n\n\ndef _is_project_output(relative, include_checkpoints):\n    parts = relative.parts\n    if any(part.startswith(\'.\') or part == \'__pycache__\' for part in parts):\n        return False\n    filename = relative.name.lower()\n    if (\'credential\' in filename or \'oauth\' in filename or filename.startswith(\'client_secret\') or\n            filename in {\'token.json\', \'tokens.json\'} or\n            any(secret in filename for secret in (\'access_token\', \'refresh_token\', \'id_token\')) or\n            filename.endswith((\'.zip\', \'.pyc\', \'.pem\', \'.key\'))):\n        return False\n    if len(parts) == 1:\n        return relative.name in ALLOWED_ROOT_FILES\n    return parts[0] in ALLOWED_FOLDERS or (include_checkpoints and parts[0] == \'checkpoints\')\n\n\ndef verify_backup(archive_file, max_total_bytes=20 * 1024**3):\n    """Check structure, file sizes and hashes before accepting a local ZIP.\n\n    Hashes detect corruption; they do not establish who created an archive.\n    Restore only backups from a source you trust.\n    """\n    with zipfile.ZipFile(archive_file) as archive:\n        members = archive.infolist()\n        names = [item.filename for item in members]\n        if len(names) != len(set(names)) or \'manifest.json\' not in names:\n            raise ValueError(\'Backup has duplicate members or no manifest.\')\n        if sum(item.file_size for item in members) > max_total_bytes:\n            raise ValueError(\'Backup exceeds the configured uncompressed-size limit.\')\n        for item in members:\n            _relative_path(item.filename)\n            if stat.S_ISLNK(item.external_attr >> 16) or item.is_dir():\n                raise ValueError(\'Links and directory entries are not valid backup members.\')\n        if archive.getinfo(\'manifest.json\').file_size > 10 * 1024**2:\n            raise ValueError(\'Backup manifest is too large.\')\n        manifest = json.loads(archive.read(\'manifest.json\'))\n        if manifest.get(\'format\') != FORMAT or not isinstance(manifest.get(\'files\'), list):\n            raise ValueError(\'Unknown backup format.\')\n        expected_names = {\'manifest.json\'}\n        paths = set()\n        for record in manifest[\'files\']:\n            relative = _relative_path(record[\'path\'])\n            if relative.as_posix() in paths:\n                raise ValueError(\'Duplicate path in backup manifest.\')\n            paths.add(relative.as_posix())\n            member_name = \'project/\' + relative.as_posix()\n            expected_names.add(member_name)\n            if member_name not in names:\n                raise ValueError(\'A manifest file is missing from the ZIP.\')\n            info = archive.getinfo(member_name)\n            if info.file_size != record[\'size_bytes\']:\n                raise ValueError(f\'Backup size check failed: {relative}\')\n            digest = hashlib.sha256()\n            with archive.open(info) as stream:\n                for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n                    digest.update(chunk)\n            if digest.hexdigest() != record[\'sha256\']:\n                raise ValueError(f\'Backup checksum check failed: {relative}\')\n        if set(names) != expected_names:\n            raise ValueError(\'Backup contains files not listed in the manifest.\')\n        # Reading every member to EOF above also checks each ZIP CRC.\n    return manifest\n\n\ndef create_backup(project_root, export_dir, step, revision, include_checkpoints=False):\n    """Archive selected project outputs and verify the completed ZIP by reading it."""\n    root = Path(project_root).resolve(strict=True)\n    export = Path(export_dir).resolve()\n    if not root.is_dir() or export.is_relative_to(root):\n        raise ValueError(\'Use an existing project directory and an export directory outside it.\')\n    if not re.fullmatch(r\'[A-Za-z0-9_-]+\', step):\n        raise ValueError(\'Use a simple step name, e.g. step0.\')\n    export.mkdir(parents=True, exist_ok=True)\n    records, selected = [], []\n    folders = ALLOWED_FOLDERS | ({\'checkpoints\'} if include_checkpoints else set())\n    candidates = [root / name for name in ALLOWED_ROOT_FILES if (root / name).is_file()]\n    for name in sorted(folders):\n        folder = root / name\n        if folder.is_symlink():\n            raise ValueError(f\'Refusing to back up a linked project directory: {name}\')\n        if folder.is_dir():\n            candidates.extend(folder.rglob(\'*\'))\n    for path in sorted(candidates):\n        relative = path.relative_to(root)\n        if not _is_project_output(relative, include_checkpoints):\n            continue\n        if path.is_symlink() or any((root / Path(*relative.parts[:i])).is_symlink()\n                                    for i in range(1, len(relative.parts))):\n            raise ValueError(f\'Refusing to back up a link: {relative}\')\n        if not path.is_file():\n            continue\n        path.resolve(strict=True).relative_to(root)\n        records.append({\'path\': relative.as_posix(), \'size_bytes\': path.stat().st_size,\n                        \'sha256\': sha256_file(path)})\n        selected.append((path, relative))\n    if not records:\n        raise ValueError(\'No project outputs were selected for backup.\')\n    created = datetime.now(timezone.utc)\n    manifest = {\'format\': FORMAT, \'step\': step, \'revision\': revision,\n                \'created_at_utc\': created.isoformat(), \'files\': records,\n                \'includes_checkpoints\': include_checkpoints}\n    name = f\'NER_{step}_{created:%Y%m%dT%H%M%SZ}_{uuid.uuid4().hex[:8]}.zip\'\n    output = export / name\n    partial = output.with_suffix(\'.partial\')\n    with zipfile.ZipFile(partial, \'x\', compression=zipfile.ZIP_DEFLATED,\n                         compresslevel=2, strict_timestamps=False) as archive:\n        archive.writestr(\'manifest.json\', json.dumps(manifest, indent=2))\n        for path, relative in selected:\n            archive.write(path, arcname=\'project/\' + relative.as_posix())\n    verify_backup(partial)\n    if output.exists():\n        raise FileExistsError(output)\n    partial.rename(output)\n    return {\'path\': str(output), \'name\': output.name, \'size_bytes\': output.stat().st_size,\n            \'sha256\': sha256_file(output), \'file_count\': len(records),\n            \'archive_verified\': True, \'download_confirmed\': False}\n\n\ndef restore_backup(archive_file, destination):\n    """Restore verified bytes into a new directory without executing any code."""\n    manifest = verify_backup(archive_file)\n    target = Path(destination).resolve()\n    if target.exists():\n        raise FileExistsError(\'Choose a new, empty destination name for restoration.\')\n    target.mkdir(parents=True, exist_ok=False)\n    with zipfile.ZipFile(archive_file) as archive:\n        for record in manifest[\'files\']:\n            relative = _relative_path(record[\'path\'])\n            path = target.joinpath(*relative.parts)\n            path.parent.mkdir(parents=True, exist_ok=True)\n            with archive.open(\'project/\' + relative.as_posix()) as source, path.open(\'xb\') as output:\n                shutil.copyfileobj(source, output, length=8 * 1024 * 1024)\n            if sha256_file(path) != record[\'sha256\']:\n                raise ValueError(f\'Restored file checksum failed: {relative}\')\n    return {\'project_root\': str(target), \'step\': manifest[\'step\'], \'file_count\': len(manifest[\'files\'])}\n', 'step1_data.py': '"""Pinned CoNLL-2003 loading and validation. Run real data work in Colab only."""\nimport hashlib\nimport json\nfrom datetime import datetime, timezone\nfrom numbers import Integral\nfrom pathlib import Path\n\nREPO_ID = \'tomaarsen/conll2003\'\nREVISION = \'3a0c0677d3177b6122fceed3239dda7a5de6a514\'\nEXPECTED_ROWS = {\'train\': 14041, \'validation\': 3250, \'test\': 3453}\nTAG_FIELDS = (\'pos_tags\', \'chunk_tags\', \'ner_tags\')\nREQUIRED_FIELDS = (\'id\', \'tokens\', *TAG_FIELDS)\nEXPECTED_NER_LABELS = {\'O\'} | {f\'{p}-{t}\' for p in (\'B\', \'I\') for t in (\'PER\', \'ORG\', \'LOC\', \'MISC\')}\nPARQUET_FILES = {s: f\'data/{s}-00000-of-00001.parquet\' for s in EXPECTED_ROWS}\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + \'.partial\')\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef load_pinned_dataset(project_root):\n    """Download public data, without auth or remote loading scripts; retain file hashes."""\n    import datasets\n    from huggingface_hub import snapshot_download\n\n    if datasets.__version__ != \'4.0.0\':\n        raise RuntimeError(\'Use datasets==4.0.0; install it and restart the Python session.\')\n    cache = Path(project_root) / \'data\' / \'hf_cache\'\n    snapshot = Path(snapshot_download(\n        repo_id=REPO_ID, repo_type=\'dataset\', revision=REVISION,\n        allow_patterns=[*PARQUET_FILES.values(), \'README.md\'],\n        cache_dir=str(cache / \'hub\'), token=False,\n    ))\n    files = {split: str(snapshot / name) for split, name in PARQUET_FILES.items()}\n    source_files = []\n    for name in [*PARQUET_FILES.values(), \'README.md\']:\n        path = snapshot / name\n        if not path.is_file():\n            raise FileNotFoundError(f\'Pinned source file missing: {name}\')\n        source_files.append({\'path\': name, \'size_bytes\': path.stat().st_size, \'sha256\': file_sha256(path)})\n    dataset = datasets.load_dataset(\'parquet\', data_files=files,\n                                    cache_dir=str(cache / \'arrow\'), token=False)\n    provenance = {\n        \'requested_source\': \'eriktks/conll2003\',\n        \'actual_source\': REPO_ID, \'revision\': REVISION, \'format\': \'parquet\',\n        \'source_url\': f\'https://huggingface.co/datasets/{REPO_ID}/tree/{REVISION}\',\n        \'reason\': \'Development document permits Parquet fallback; datasets 4.0 removes loading scripts.\',\n        \'license_as_stated_by_card\': \'other\',\n        \'source_files\': source_files,\n        \'preprocessing\': \'None: no shuffle, split, filtering, label conversion or token normalization.\',\n        \'parity_scope\': \'Split sizes, metadata, all-row structure/IOB2 and ordered content hashes checked. \'\n                        \'Byte-for-byte equivalence to eriktks/conll2003 has not been established.\',\n        \'recorded_at_utc\': datetime.now(timezone.utc).isoformat(),\n    }\n    return dataset, provenance\n\n\ndef label_names(split):\n    names = {}\n    for field in TAG_FIELDS:\n        feature = split.features[field]\n        values = getattr(getattr(feature, \'feature\', None), \'names\', None)\n        if not values or len(values) != len(set(values)):\n            raise ValueError(f\'{field} must have unique ClassLabel names in feature metadata.\')\n        names[field] = list(values)\n    if set(names[\'ner_tags\']) != EXPECTED_NER_LABELS:\n        raise ValueError(\'NER metadata must contain O and B/I labels for PER, ORG, LOC, MISC.\')\n    return names\n\n\ndef audit_dataset(dataset, expected_rows=None):\n    """Reject corrupt/IOB1 data instead of silently repairing labels or dropping rows.\n\n    expected_rows is injectable for tiny offline fixtures; real runs use EXPECTED_ROWS.\n    Hash input: UTF-8 compact JSON for each core record, decoded tag strings, plus LF.\n    """\n    expected_rows = EXPECTED_ROWS if expected_rows is None else expected_rows\n    if set(dataset) != set(expected_rows):\n        raise ValueError(\'Unexpected dataset splits.\')\n    report = {\'splits\': {}, \'label_names\': None, \'iob2_violations\': 0,\n              \'hash_format\': \'core-fields-decoded-tags-jsonl-utf8-v1\'}\n    for split_name, expected in expected_rows.items():\n        split = dataset[split_name]\n        if len(split) != expected:\n            raise ValueError(f\'{split_name}: expected {expected} rows, received {len(split)}.\')\n        if not set(REQUIRED_FIELDS).issubset(split.column_names):\n            raise ValueError(f\'{split_name}: missing required columns.\')\n        names = label_names(split)\n        if report[\'label_names\'] is None:\n            report[\'label_names\'] = names\n        elif names != report[\'label_names\']:\n            raise ValueError(f\'{split_name}: label metadata differs between splits.\')\n        has_documents = {\'document_id\', \'sentence_id\'}.issubset(split.column_names)\n        digest = hashlib.sha256()\n        tokens_total = 0\n        previous_doc, previous_sentence = None, None\n        document_count = 0\n        for index, row in enumerate(split):\n            where = f\'{split_name}[{index}]\'\n            if row[\'id\'] != str(index):\n                raise ValueError(f\'{where}: id/order changed (expected id={index}).\')\n            tokens = row[\'tokens\']\n            if not isinstance(tokens, list) or not tokens or any(not isinstance(t, str) or not t for t in tokens):\n                raise ValueError(f\'{where}: empty or invalid token list.\')\n            if \'-DOCSTART-\' in tokens:\n                raise ValueError(f\'{where}: unexpected DOCSTART sentinel.\')\n            decoded = {}\n            for field in TAG_FIELDS:\n                tags = row[field]\n                if not isinstance(tags, list) or len(tags) != len(tokens):\n                    raise ValueError(f\'{where}: token/{field} length mismatch.\')\n                if any(isinstance(t, bool) or not isinstance(t, Integral) or not 0 <= t < len(names[field]) for t in tags):\n                    raise ValueError(f\'{where}: invalid {field} index.\')\n                decoded[field] = [names[field][int(t)] for t in tags]\n            previous = \'O\'\n            for position, tag in enumerate(decoded[\'ner_tags\']):\n                if tag.startswith(\'I-\') and previous not in (\'B-\' + tag[2:], tag):\n                    raise ValueError(f\'{where} token {position}: illegal IOB2 transition {previous} -> {tag}.\')\n                previous = tag\n            if has_documents:\n                doc, sentence = row[\'document_id\'], row[\'sentence_id\']\n                if any(isinstance(v, bool) or not isinstance(v, Integral) for v in (doc, sentence)):\n                    raise ValueError(f\'{where}: invalid document/sentence id.\')\n                if previous_doc is None or doc != previous_doc:\n                    if sentence != 0 or (previous_doc is not None and doc != previous_doc + 1):\n                        raise ValueError(f\'{where}: document order is not contiguous.\')\n                    document_count += 1\n                elif sentence != previous_sentence + 1:\n                    raise ValueError(f\'{where}: sentence order is not contiguous.\')\n                previous_doc, previous_sentence = doc, sentence\n            record = {\'id\': row[\'id\'], \'tokens\': tokens, **decoded}\n            digest.update((json.dumps(record, ensure_ascii=False, separators=(\',\', \':\')) + \'\\n\').encode(\'utf-8\'))\n            tokens_total += len(tokens)\n        report[\'splits\'][split_name] = {\n            \'rows\': len(split), \'tokens\': tokens_total, \'ordered_core_sha256\': digest.hexdigest(),\n            \'id_order_verified\': True, \'document_order_verified\': has_documents,\n            \'documents\': document_count if has_documents else None,\n        }\n    return report\n\n\ndef save_and_verify(dataset, project_root, run_id, report):\n    """Keep Arrow data on the runtime; verify persistence without putting data in ZIPs."""\n    from datasets import load_from_disk\n\n    destination = Path(project_root) / \'data\' / f\'conll2003_{run_id}\'\n    if destination.exists():\n        raise FileExistsError(\'This run was already saved. Rerun initialization for a new run_id.\')\n    dataset.save_to_disk(str(destination))\n    restored = load_from_disk(str(destination))\n    check = audit_dataset(restored)\n    if check != report:\n        raise ValueError(\'Saved dataset differs after read-back.\')\n    # Includes auxiliary columns and feature metadata, in addition to ordered core hashes.\n    files = [{\'path\': p.relative_to(destination).as_posix(), \'size_bytes\': p.stat().st_size,\n              \'sha256\': file_sha256(p)} for p in sorted(destination.rglob(\'*\')) if p.is_file()]\n    return {\'path\': str(destination), \'read_back_verified\': True, \'files\': files}\n'}
for filename, source in SOURCE_BUNDLE.items():
    (PROJECT_ROOT / 'src' / filename).write_text(source, encoding='utf-8')

def import_project_module(name, filename):
    spec = importlib.util.spec_from_file_location(name, PROJECT_ROOT / 'src' / filename)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

backup = import_project_module('ner_local_backup', 'local_backup.py')
data_tools = import_project_module('ner_step1_data', 'step1_data.py')
print('Backup and dataset validation tools ready.')


## 1.3 下载固定版本数据
从公开 Hugging Face 仓库下载约 2 MB Parquet 数据。首次运行需要网络；无需 Hugging Face 登录。
下载完成会记录每个文件的 SHA-256。若下载失败，请反馈错误，不更换未知来源或填写 API 密钥。


In [ ]:
STEP1_COMPLETE = DATA_LOADED = DATA_VERIFIED = DATA_SAVED = BACKUP_VERIFIED = BACKUP_DOWNLOADED = False
raw_datasets, SOURCE_RECORD = data_tools.load_pinned_dataset(PROJECT_ROOT)
DATA_LOADED = True
print('Source:', SOURCE_RECORD['actual_source'])
print('Revision:', SOURCE_RECORD['revision'])
print(raw_datasets)


## 1.4 全量校验并显示样例
检查三个划分的全部句子。NER 的 I- 标签必须跟在同类型 B-/I- 后面；遇到异常立即停止，
不自动“修正”数据。标签编号从 features 读取；展示前三个训练句的词和标签字符串。


In [ ]:
STEP1_COMPLETE = DATA_VERIFIED = DATA_SAVED = BACKUP_VERIFIED = BACKUP_DOWNLOADED = False
if not DATA_LOADED:
    raise RuntimeError('Complete 1.3 first.')
AUDIT = data_tools.audit_dataset(raw_datasets)
LABEL_NAMES = AUDIT['label_names']['ner_tags']
id2label = dict(enumerate(LABEL_NAMES))
label2id = {label: index for index, label in id2label.items()}
DATA_VERIFIED = True
for split, record in AUDIT['splits'].items():
    print(f"{split:10s} rows={record['rows']:5d}  tokens={record['tokens']:6d}  order=OK  IOB2=OK")
print('NER labels from metadata:', id2label)
for index in range(3):
    sample = raw_datasets['train'][index]
    print(f"\nTRAIN SAMPLE {index}, id={sample['id']}")
    for token, tag in zip(sample['tokens'], sample['ner_tags']):
        print(f'{token:20s} {id2label[tag]}')
print('DATA CHECKS PASSED. Continue to 1.5 to save and back up.')


## 1.5 保存数据、来源和验收结果，生成 ZIP
将数据保存为运行时 Arrow 文件并读回核验，供第 2 步使用。ZIP 包含源码、文件哈希、
依赖版本、动态标签映射和校验记录；排除数据集和缓存。新运行时可按固定版本重新下载数据。


In [ ]:
STEP1_COMPLETE = DATA_SAVED = BACKUP_VERIFIED = BACKUP_DOWNLOADED = False
if not DATA_VERIFIED:
    raise RuntimeError('Complete 1.4 before saving.')
SAVED_DATA = data_tools.save_and_verify(raw_datasets, PROJECT_ROOT, RUN_ID, AUDIT)
package_names = ['datasets', 'huggingface-hub', 'pyarrow', 'numpy', 'pandas', 'fsspec']
RUNTIME_RECORD = {
    'notebook_revision': NOTEBOOK_REVISION, 'run_id': RUN_ID, 'seed': SEED,
    'python': sys.version, 'platform': platform.platform(),
    'packages': {name: metadata.version(name) for name in package_names},
    'storage': 'runtime-local-with-manual-download',
}
data_tools.write_json(PROJECT_ROOT / 'logs' / f'step1_environment_{RUN_ID}.json', RUNTIME_RECORD)
data_tools.write_json(PROJECT_ROOT / 'config' / f'dataset_source_{RUN_ID}.json', SOURCE_RECORD)
data_tools.write_json(PROJECT_ROOT / 'config' / 'label_mapping.json',
                      {'id2label': id2label, 'label2id': label2id, 'all_tag_names': AUDIT['label_names']})
data_tools.write_json(PROJECT_ROOT / 'results' / f'step1_validation_{RUN_ID}.json', AUDIT)
data_tools.write_json(PROJECT_ROOT / 'config' / 'dataset_latest.json', {
    'run_id': RUN_ID, 'source': SOURCE_RECORD, 'saved_data': SAVED_DATA,
    'audit': AUDIT, 'status': 'data_verified_download_not_confirmed',
})
(PROJECT_ROOT / 'config' / 'requirements-data-observed.txt').write_text(
    ''.join(f'{name}=={version}\n' for name, version in RUNTIME_RECORD['packages'].items()), encoding='utf-8')
if not (PROJECT_ROOT / 'README.md').exists():
    (PROJECT_ROOT / 'README.md').write_text(
        'NER Colab project. Keep the original downloaded ZIP and notebook on your computer.\n'
        'Data/cache excluded from ZIP; rerun Step 1 to download the pinned data.\n'
        'Restore your trusted ZIP with local_backup.restore_backup into a NEW directory.\n'
        'Restoration does not reinstall dependencies or resume a Python session.\n', encoding='utf-8')
DATA_SAVED = True
print('Data saved and read-back verified:', SAVED_DATA['path'])


In [ ]:
STEP1_COMPLETE = BACKUP_VERIFIED = BACKUP_DOWNLOADED = False
if not DATA_SAVED:
    raise RuntimeError('Save and verify data in the previous cell first.')
BACKUP_RECEIPT = backup.create_backup(PROJECT_ROOT, EXPORT_DIR, step='step1',
                                        revision=NOTEBOOK_REVISION, include_checkpoints=False)
BACKUP_PATH = Path(BACKUP_RECEIPT['path'])
BACKUP_VERIFIED = BACKUP_RECEIPT['archive_verified']
print('ZIP READY — computer download still needs confirmation.')
print(json.dumps(BACKUP_RECEIPT, indent=2))


In [ ]:
if not BACKUP_VERIFIED or not BACKUP_PATH.is_file():
    raise RuntimeError('Generate the verified ZIP first.')
try:
    from google.colab import files
    files.download(str(BACKUP_PATH))
    print('Download requested:', BACKUP_PATH.name)
except Exception as exc:
    print('Automatic download did not finish:', type(exc).__name__, str(exc))
print('If needed, use the Colab Files panel to download:', BACKUP_PATH)
print('Check the computer copy, and save the notebook using File > Download .ipynb.')


## 1.6 确认电脑备份并反馈
确认电脑上已有同名、同字节数 ZIP 后，把 `BACKUP_DOWNLOADED` 改成 `True` 并运行。
将最后的验收输出发给我；如果出错，提供小节编号和完整错误文本。
本步通过后再进行第 2 步 EDA。


In [ ]:
BACKUP_DOWNLOADED = False  # Only set True after checking your computer copy.
STEP1_COMPLETE = False
if not (DATA_VERIFIED and DATA_SAVED and BACKUP_VERIFIED):
    raise RuntimeError('Complete 1.3–1.5 first.')
if not BACKUP_DOWNLOADED:
    print('WAITING FOR DOWNLOAD CONFIRMATION: check the computer ZIP, then set True.')
else:
    # Recheck the archive and its receipt before issuing a completion message.
    backup.verify_backup(BACKUP_PATH)
    if backup.sha256_file(BACKUP_PATH) != BACKUP_RECEIPT['sha256']:
        raise RuntimeError('The ZIP changed after its receipt was generated.')
    STEP1_COMPLETE = True
    print('STEP 1 COMPLETE: data and ZIP verified; computer download confirmed by the user.')
    print('Rows:', {split: row['rows'] for split, row in AUDIT['splits'].items()})
    print('IOB2 violations:', AUDIT['iob2_violations'])
    print('NER labels:', LABEL_NAMES)
    print('Source:', SOURCE_RECORD['actual_source'], SOURCE_RECORD['revision'])
    print('Saved data:', SAVED_DATA['path'])
    print('Backup:', BACKUP_RECEIPT['name'])
    print('Bytes:', BACKUP_RECEIPT['size_bytes'])
    print('SHA-256:', BACKUP_RECEIPT['sha256'])
    print('Send this output before Step 2.')
